- Ce script automatise le téléchargement d'images satellites à partir de coordonnées GPS des ménages de l'EHCVM 2021 fournies dans un fichier CSV, appelé Data_EHCVM_2021.csv en utilisant l'API de Google Maps. Il est conçu pour extraire des images de type satellite avec un niveau de zoom spécifique et une résolution de 400x400 pixels, idéales pour des analyses géospatiales. Ces images sont enregistrées au format JPEG dans un dossier local dont le chemin est défini dynamiquement, garantissant un classement par niveau de zoom et année pour une organisation optimale des données.

- Le processus commence par la création du dossier de destination si celui-ci n'existe pas déjà. Le script lit ensuite le fichier CSV, qui contient les coordonnées de chaque point d'intérêt, pour chaque paire de latitude et longitude. Une fonction dédiée de téléchargement est utilisée pour chaque image, intégrant un mécanisme de réessai avec backoff exponentiel pour gérer les interruptions de réseau. Ce mécanisme améliore la résilience du script, permettant de réessayer les téléchargements en cas de perte de connexion ou de délais d'attente dépassés.

- Une fois la requête envoyée, le script vérifie le statut de la réponse pour s'assurer que l'image a bien été récupérée. En cas de succès, elle est enregistrée localement avec un nom de fichier basé sur les coordonnées GPS et le niveau de zoom, tandis qu’en cas d’échec, des messages d’erreur sont affichés pour informer l’utilisateur et initier de nouvelles tentatives si nécessaire. Une barre de progression, rendue possible par la bibliothèque tqdm, permet de visualiser en temps réel l'avancement du téléchargement, offrant un suivi pratique pour des traitements de lots de grande taille.

In [1]:
# Installer les packages nécessaires
!pip install requests tqdm

In [2]:
import requests
import os
import time
import pandas as pd
from tqdm import tqdm

In [3]:
# Paramètres
api_key = ""  # Clé API de Février
zoom = 18  # Niveau de zoom désiré
image_format = "jpeg"  # Format de l'image
folder_path = f"D:\\wealth_predict_2021\\data\\downloaded\\Image_satellite_EHCVM_2021_Zoom_{zoom}_Image_2024"


In [4]:
# Création du dossier si non existant
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

# Fonction pour télécharger une image satellite avec un mécanisme de réessai en cas de connexion perdue
def download_satellite_image(latitude, longitude, zoom, api_key, folder_path, image_format="jpeg", retries=3):
    # Nom du fichier basé sur les coordonnées et le zoom
    file_name = f"lat_{latitude}_lon_{longitude}_zoom_{zoom}.{image_format}"
    file_path = os.path.join(folder_path, file_name)

    # Vérifier si l'image existe déjà
    if os.path.exists(file_path):
        print(f"L'image existe déjà : {file_name}")
        return False  # Indique qu'aucun téléchargement n'a été effectué

    # Paramètres pour l'API Static Maps
    size = "400x400"  # Taille de l'image en pixels
    map_type = "satellite"
    url = f"https://maps.googleapis.com/maps/api/staticmap?center={latitude},{longitude}&zoom={zoom}&size={size}&maptype={map_type}&key={api_key}"

    # Boucle de réessai avec backoff exponentiel
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=30)  # Augmente le délai de timeout à 30 secondes
            if response.status_code == 200:
                # Enregistrer l'image
                with open(file_path, 'wb') as file:
                    file.write(response.content)
                return True  # Indique que le téléchargement a été effectué avec succès
            else:
                print(f"Erreur lors du téléchargement de l'image ({latitude}, {longitude}) : {response.status_code}")
                return False
        except requests.exceptions.ReadTimeout:
            print(f"Timeout lors du téléchargement de l'image ({latitude}, {longitude}). Nouvelle tentative...")
            time.sleep(2 ** attempt)  # Backoff exponentiel
        except requests.ConnectionError:
            print(f"Connexion perdue pour l'image ({latitude}, {longitude}). Nouvelle tentative dans 1 minutes...")
            time.sleep(60)  # Attendre 1 minutes en cas de problème de connexion
    return False  # Si tous les essais échouent



In [5]:
# Charger le fichier CSV
csv_path = r'D:\wealth_predict_2021\data\original_csv_file\Data_EHCVM_2021.csv'
df = pd.read_csv(csv_path)

# Boucle de téléchargement avec barre de progression et gestion des interruptions de connexion
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Téléchargement des images"):
    latitude = row['gps__latitude']
    longitude = row['gps__longitude']
    
    while True:
        result = download_satellite_image(latitude, longitude, zoom, api_key, folder_path, image_format)
        
        if result == True:
            break  # Téléchargement réussi, on passe à l'image suivante
        elif result == False:
            break  # L'image existe déjà ou il y a eu une autre erreur, on passe à l'image suivante


Téléchargement des images:   1%|▎                                                  | 95/12952 [00:00<00:14, 865.58it/s]

L'image existe déjà : lat_5.3397445_lon_-4.0252938_zoom_18.jpeg
L'image existe déjà : lat_5.34098257_lon_-4.02586833_zoom_18.jpeg
L'image existe déjà : lat_5.33999731_lon_-4.02544883_zoom_18.jpeg
L'image existe déjà : lat_5.34002433_lon_-4.02578813_zoom_18.jpeg
L'image existe déjà : lat_5.34090143_lon_-4.02588236_zoom_18.jpeg
L'image existe déjà : lat_5.34012863_lon_-4.02590759_zoom_18.jpeg
L'image existe déjà : lat_5.34028756_lon_-4.0258872_zoom_18.jpeg
L'image existe déjà : lat_5.34026997_lon_-4.02590191_zoom_18.jpeg
L'image existe déjà : lat_5.33975644_lon_-4.02589581_zoom_18.jpeg
L'image existe déjà : lat_5.34086613_lon_-4.02604911_zoom_18.jpeg
L'image existe déjà : lat_5.34022063_lon_-4.02579428_zoom_18.jpeg
L'image existe déjà : lat_5.3408854_lon_-4.0257857_zoom_18.jpeg
L'image existe déjà : lat_5.3356416_lon_-4.0269785_zoom_18.jpeg
L'image existe déjà : lat_5.33700177_lon_-4.02586289_zoom_18.jpeg
L'image existe déjà : lat_5.33464472_lon_-4.02636608_zoom_18.jpeg
L'image existe dé

Téléchargement des images:   3%|█▎                                               | 338/12952 [00:00<00:12, 1043.95it/s]

L'image existe déjà : lat_5.34927214_lon_-4.02924859_zoom_18.jpeg
L'image existe déjà : lat_5.34906577_lon_-4.02956454_zoom_18.jpeg
L'image existe déjà : lat_5.30385718_lon_-3.99737433_zoom_18.jpeg
L'image existe déjà : lat_5.30364723_lon_-3.99778122_zoom_18.jpeg
L'image existe déjà : lat_5.30364329_lon_-3.99687219_zoom_18.jpeg
L'image existe déjà : lat_5.3038362_lon_-3.9980271_zoom_18.jpeg
L'image existe déjà : lat_5.3046947_lon_-3.99857874_zoom_18.jpeg
L'image existe déjà : lat_5.30393629_lon_-3.99632357_zoom_18.jpeg
L'image existe déjà : lat_5.3034431_lon_-3.99795788_zoom_18.jpeg
L'image existe déjà : lat_5.30390384_lon_-3.9980529_zoom_18.jpeg
L'image existe déjà : lat_5.30408099_lon_-3.99631246_zoom_18.jpeg
L'image existe déjà : lat_5.30392923_lon_-3.99679067_zoom_18.jpeg
L'image existe déjà : lat_5.30405311_lon_-3.99825368_zoom_18.jpeg
L'image existe déjà : lat_5.30421508_lon_-3.99850136_zoom_18.jpeg
L'image existe déjà : lat_5.44117162_lon_-4.04775842_zoom_18.jpeg
L'image existe 

Téléchargement des images:   5%|██▏                                              | 585/12952 [00:00<00:11, 1115.67it/s]

...

Téléchargement des images:  56%|█████████████████████████▋                    | 7228/12952 [1:59:29<1:54:19,  1.20s/it]

L'image existe déjà : lat_6.1423572_lon_-5.9518134_zoom_18.jpeg


Téléchargement des images:  94%|██████████████████████████████████████████▏  | 12158/12952 [3:52:55<4:26:47, 20.16s/it]

Connexion perdue pour l'image (5.76456866, -6.51233672). Nouvelle tentative dans 1 minutes...


Téléchargement des images: 100%|███████████████████████████████████████████████| 12952/12952 [4:11:11<00:00,  1.16s/it]
